In [ ]:
%%sql -r dataframe_1
-- =====================================================
-- PHASE 5A: OPS TABLES
-- NOTEBOOK: NB_05_DATA_QUALITY_RECONCILIATION
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA OPS;

-- 1. Ingestion audit
CREATE TABLE IF NOT EXISTS OPS.INGESTION_AUDIT (
    AUDIT_ID              NUMBER AUTOINCREMENT,
    BATCH_ID              VARCHAR(100),
    SOURCE_TYPE           VARCHAR(50),
    SOURCE_FILE_NAME      VARCHAR(500),
    FILE_TIMESTAMP        TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    RECORD_COUNT          NUMBER(38,0),
    SUCCESS_COUNT         NUMBER(38,0),
    REJECTED_COUNT        NUMBER(38,0),
    STATUS                VARCHAR(30),
    ERROR_MESSAGE         VARCHAR(2000)
);

-- 2. Batch control
CREATE TABLE IF NOT EXISTS OPS.BATCH_CONTROL (
    BATCH_ID              VARCHAR(100),
    SOURCE_TYPE           VARCHAR(50),
    SOURCE_FILE_NAME      VARCHAR(500),
    FILE_TIMESTAMP        TIMESTAMP_NTZ,
    BATCH_START_TIMESTAMP TIMESTAMP_NTZ,
    BATCH_END_TIMESTAMP   TIMESTAMP_NTZ,
    STATUS                VARCHAR(30),
    RECORD_COUNT          NUMBER(38,0),
    PROCESSED_COUNT       NUMBER(38,0),
    REJECTED_COUNT        NUMBER(38,0)
);

-- 3. Rejected member records
CREATE TABLE IF NOT EXISTS OPS.REJECTED_MEMBER_RECORDS (
    REJECT_ID             NUMBER AUTOINCREMENT,
    BATCH_ID              VARCHAR(100),
    SOURCE_FILE_NAME      VARCHAR(500),
    SOURCE_ROW_NUMBER     NUMBER(38,0),
    RAW_RECORD            VARIANT,
    REJECTION_REASON      VARCHAR(2000),
    REJECTION_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- 4. Rejected redemption records
CREATE TABLE IF NOT EXISTS OPS.REJECTED_REDEMPTION_RECORDS (
    REJECT_ID             NUMBER AUTOINCREMENT,
    BATCH_ID              VARCHAR(100),
    SOURCE_FILE_NAME      VARCHAR(500),
    SOURCE_RECORD_NUMBER  NUMBER(38,0),
    RAW_RECORD            VARIANT,
    REJECTION_REASON      VARCHAR(2000),
    REJECTION_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- 5. Data quality results
CREATE TABLE IF NOT EXISTS OPS.DATA_QUALITY_RESULTS (
    RESULT_ID             NUMBER AUTOINCREMENT,
    BATCH_ID              VARCHAR(100),
    SOURCE_TYPE           VARCHAR(50),
    TABLE_NAME            VARCHAR(255),
    RULE_NAME             VARCHAR(255),
    TOTAL_RECORDS         NUMBER(38,0),
    PASSED_RECORDS        NUMBER(38,0),
    FAILED_RECORDS        NUMBER(38,0),
    CHECK_STATUS          VARCHAR(30),
    EXECUTION_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Verify
SHOW TABLES IN SCHEMA OPS;

In [ ]:
%%sql -r dataframe_2
-- =====================================================
-- PHASE 5B: MART DIMENSIONAL TABLES
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA MART;

-- 1. Date dimension
CREATE TABLE IF NOT EXISTS MART.DIM_DATE (
    DATE_KEY          NUMBER(8,0),
    FULL_DATE         DATE,
    DAY_NUMBER        NUMBER(2,0),
    DAY_NAME          VARCHAR(20),
    WEEK_NUMBER       NUMBER(2,0),
    MONTH_NUMBER      NUMBER(2,0),
    MONTH_NAME        VARCHAR(20),
    QUARTER_NUMBER    NUMBER(1,0),
    YEAR_NUMBER      NUMBER(4,0),
    IS_WEEKEND        BOOLEAN
);

-- 2. Member dimension
CREATE TABLE IF NOT EXISTS MART.DIM_MEMBER (
    MEMBER_KEY          NUMBER AUTOINCREMENT,
    MEMBER_ID           VARCHAR(18),
    MEMBER_NAME         VARCHAR(255),
    ENROLLMENT_DATE     DATE,
    LAST_FLIGHT_DATE    DATE,
    TIER_CODE           VARCHAR(5),
    STATE               VARCHAR(5),
    COUNTRY             VARCHAR(5),
    AGE                 NUMBER(3,0),
    ACTIVE_MEMBER       CHAR(1),
    STALE_MEMBER        BOOLEAN,
    SOURCE_FILE_NAME    VARCHAR(500),
    FILE_TIMESTAMP      TIMESTAMP_NTZ
);

-- 3. Redemption fact
CREATE TABLE IF NOT EXISTS MART.FACT_REDEMPTION (
    REDEMPTION_KEY       NUMBER AUTOINCREMENT,
    DATE_KEY             NUMBER(8,0),
    MEMBER_KEY           NUMBER,
    MEMBER_ID            VARCHAR(18),
    TXN_ID               VARCHAR(100),
    TXN_DATE             DATE,
    PARTNER              VARCHAR(255),
    MILES_REDEEMED       NUMBER(18,0),
    STATUS               VARCHAR(50),
    SOURCE_FILE_NAME     VARCHAR(500),
    BATCH_ID             VARCHAR(100)
);

-- Verify
SHOW TABLES IN SCHEMA MART;

DESCRIBE TABLE MART.DIM_DATE;
DESCRIBE TABLE MART.DIM_MEMBER;
DESCRIBE TABLE MART.FACT_REDEMPTION;

In [ ]:
-- CELL 3: END-TO-END DATA RECONCILIATION

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

-- 1. MEMBER RECORD RECONCILIATION
SELECT
    'MEMBER' AS DATASET,
    'RAW' AS LAYER,
    COUNT(*) AS RECORD_COUNT
FROM RAW.RAW_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME LIKE '%MEMBER_PROFILE_20261125_193100.csv%'

UNION ALL

SELECT
    'MEMBER',
    'STG',
    COUNT(*)
FROM STG.STG_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME LIKE '%MEMBER_PROFILE_20261125_193100.csv%'

UNION ALL

SELECT
    'MEMBER',
    'CORE',
    COUNT(*)
FROM CORE.MEMBER_PROFILE
WHERE SOURCE_FILE_NAME LIKE '%MEMBER_PROFILE_20261125_193100.csv%';


-- 2. REDEMPTION RECORD RECONCILIATION
-- RAW counts JSON documents; STG and CORE count flattened transactions.
SELECT
    'REDEMPTION' AS DATASET,
    'RAW_JSON_DOCUMENTS' AS LAYER,
    COUNT(*) AS RECORD_COUNT
FROM RAW.RAW_REDEMPTIONS
WHERE SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%'

UNION ALL

SELECT
    'REDEMPTION',
    'STG_TRANSACTIONS',
    COUNT(*)
FROM STG.STG_REDEMPTION_TRANSACTIONS
WHERE SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%'

UNION ALL

SELECT
    'REDEMPTION',
    'CORE_TRANSACTIONS',
    COUNT(*)
FROM CORE.REDEMPTION_TRANSACTIONS
WHERE SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%';


-- 3. MEMBER VALIDATION SUMMARY
SELECT
    VALIDATION_STATUS,
    COUNT(*) AS RECORD_COUNT
FROM STG.STG_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME LIKE '%MEMBER_PROFILE_20261125_193100.csv%'
GROUP BY VALIDATION_STATUS;


-- 4. REDEMPTION VALIDATION AND BUSINESS SUMMARY
SELECT
    COUNT(*) AS TOTAL_TRANSACTIONS,
    COUNT_IF(VALIDATION_STATUS = 'VALID') AS VALID_TRANSACTIONS,
    COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS REJECTED_TRANSACTIONS,
    SUM(MILES_REDEEMED) AS TOTAL_MILES_REDEEMED,
    COUNT(DISTINCT MEMBER_ID) AS DISTINCT_MEMBERS
FROM STG.STG_REDEMPTION_TRANSACTIONS
WHERE SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%';


-- 5. COUNTRY-WISE MEMBER DISTRIBUTION
SELECT
    COUNTRY,
    COUNT(*) AS MEMBER_COUNT
FROM CORE.MEMBER_PROFILE
GROUP BY COUNTRY
ORDER BY COUNTRY;


-- 6. REDEMPTION TO MEMBER MATCHING
SELECT
    COUNT(*) AS TOTAL_TRANSACTIONS,
    COUNT_IF(M.MEMBER_ID IS NOT NULL) AS MATCHED_TRANSACTIONS,
    COUNT_IF(M.MEMBER_ID IS NULL) AS UNMATCHED_TRANSACTIONS
FROM CORE.REDEMPTION_TRANSACTIONS R
LEFT JOIN CORE.MEMBER_PROFILE M
    ON R.MEMBER_ID = M.MEMBER_ID
WHERE R.SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%';

In [ ]:
DESC TABLE OPS.DATA_QUALITY_RESULTS;

In [ ]:
-- CELL 4: STORE DATA QUALITY RESULTS

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

BEGIN TRANSACTION;

-- Remove previous results for these two batches to prevent duplicates
DELETE FROM OPS.DATA_QUALITY_RESULTS
WHERE BATCH_ID IN (
    '75e747f3112fe0f40d2adaaecf4cfa3f',
    '36dd50fa5adc4db81a373c1e7ec75cf1'
);

INSERT INTO OPS.DATA_QUALITY_RESULTS (
    BATCH_ID,
    SOURCE_TYPE,
    TABLE_NAME,
    RULE_NAME,
    TOTAL_RECORDS,
    PASSED_RECORDS,
    FAILED_RECORDS,
    CHECK_STATUS
)

WITH CHECKS AS (

    -- 1. Member mandatory fields
    SELECT
        '75e747f3112fe0f40d2adaaecf4cfa3f' AS BATCH_ID,
        'MEMBER' AS SOURCE_TYPE,
        'STG.STG_MEMBER_PROFILES' AS TABLE_NAME,
        'MANDATORY_FIELDS' AS RULE_NAME,
        COUNT(*) AS TOTAL_RECORDS,
        COUNT_IF(VALIDATION_STATUS = 'VALID') AS PASSED_RECORDS,
        COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS FAILED_RECORDS
    FROM STG.STG_MEMBER_PROFILES
    WHERE BATCH_ID = '75e747f3112fe0f40d2adaaecf4cfa3f'

    UNION ALL

    -- 2. Member ID uniqueness
    SELECT
        '75e747f3112fe0f40d2adaaecf4cfa3f',
        'MEMBER',
        'STG.STG_MEMBER_PROFILES',
        'MEMBER_ID_UNIQUENESS',
        COUNT(*),
        COUNT(DISTINCT MEMBER_ID),
        COUNT(*) - COUNT(DISTINCT MEMBER_ID)
    FROM STG.STG_MEMBER_PROFILES
    WHERE BATCH_ID = '75e747f3112fe0f40d2adaaecf4cfa3f'
      AND VALIDATION_STATUS = 'VALID'

    UNION ALL

    -- 3. Member name uniqueness (assessment key column)
    SELECT
        '75e747f3112fe0f40d2adaaecf4cfa3f',
        'MEMBER',
        'STG.STG_MEMBER_PROFILES',
        'MEMBER_NAME_UNIQUENESS',
        COUNT(*),
        COUNT(DISTINCT MEMBER_NAME),
        COUNT(*) - COUNT(DISTINCT MEMBER_NAME)
    FROM STG.STG_MEMBER_PROFILES
    WHERE BATCH_ID = '75e747f3112fe0f40d2adaaecf4cfa3f'
      AND VALIDATION_STATUS = 'VALID'

    UNION ALL

    -- 4. Supported country codes
    SELECT
        '75e747f3112fe0f40d2adaaecf4cfa3f',
        'MEMBER',
        'STG.STG_MEMBER_PROFILES',
        'COUNTRY_CODE_VALIDATION',
        COUNT(*),
        COUNT_IF(COUNTRY IN ('IND', 'USA', 'PHIL', 'CAN', 'AU')),
        COUNT_IF(COUNTRY IS NULL OR COUNTRY NOT IN ('IND', 'USA', 'PHIL', 'CAN', 'AU'))
    FROM STG.STG_MEMBER_PROFILES
    WHERE BATCH_ID = '75e747f3112fe0f40d2adaaecf4cfa3f'
      AND VALIDATION_STATUS = 'VALID'

    UNION ALL

    -- 5. Redemption mandatory fields
    SELECT
        '36dd50fa5adc4db81a373c1e7ec75cf1',
        'REDEMPTION',
        'STG.STG_REDEMPTION_TRANSACTIONS',
        'MANDATORY_FIELDS',
        COUNT(*),
        COUNT_IF(VALIDATION_STATUS = 'VALID'),
        COUNT_IF(VALIDATION_STATUS = 'REJECTED')
    FROM STG.STG_REDEMPTION_TRANSACTIONS
    WHERE BATCH_ID = '36dd50fa5adc4db81a373c1e7ec75cf1'

    UNION ALL

    -- 6. Redemption transaction-key uniqueness
    SELECT
        '36dd50fa5adc4db81a373c1e7ec75cf1',
        'REDEMPTION',
        'STG.STG_REDEMPTION_TRANSACTIONS',
        'TRANSACTION_KEY_UNIQUENESS',
        COUNT(*),
        COUNT(DISTINCT MEMBER_ID || '|' || TXN_ID),
        COUNT(*) - COUNT(DISTINCT MEMBER_ID || '|' || TXN_ID)
    FROM STG.STG_REDEMPTION_TRANSACTIONS
    WHERE BATCH_ID = '36dd50fa5adc4db81a373c1e7ec75cf1'
      AND VALIDATION_STATUS = 'VALID'

    UNION ALL

    -- 7. Redemption member matching
    SELECT
        '36dd50fa5adc4db81a373c1e7ec75cf1',
        'REDEMPTION',
        'CORE.REDEMPTION_TRANSACTIONS',
        'MEMBER_REFERENCE_VALIDATION',
        COUNT(*),
        COUNT_IF(M.MEMBER_ID IS NOT NULL),
        COUNT_IF(M.MEMBER_ID IS NULL)
    FROM CORE.REDEMPTION_TRANSACTIONS R
    LEFT JOIN CORE.MEMBER_PROFILE M
        ON R.MEMBER_ID = M.MEMBER_ID
    WHERE R.BATCH_ID = '36dd50fa5adc4db81a373c1e7ec75cf1'
)

SELECT
    BATCH_ID,
    SOURCE_TYPE,
    TABLE_NAME,
    RULE_NAME,
    TOTAL_RECORDS,
    PASSED_RECORDS,
    FAILED_RECORDS,
    CASE
        WHEN FAILED_RECORDS = 0 THEN 'PASSED'
        ELSE 'FAILED'
    END
FROM CHECKS;

COMMIT;

-- Verify persisted results
SELECT *
FROM OPS.DATA_QUALITY_RESULTS
WHERE BATCH_ID IN (
    '75e747f3112fe0f40d2adaaecf4cfa3f',
    '36dd50fa5adc4db81a373c1e7ec75cf1'
)
ORDER BY SOURCE_TYPE, RULE_NAME;

In [ ]:
-- CELL 5: VERIFY OPERATIONAL AUDIT TABLES

USE DATABASE SKYPOINTS_INCU;

DESC TABLE OPS.INGESTION_AUDIT;


In [ ]:
USE DATABASE SKYPOINTS_INCU;

DESC TABLE OPS.BATCH_CONTROL;


In [ ]:
USE DATABASE SKYPOINTS_INCU;

DESC TABLE OPS.REJECTED_MEMBER_RECORDS;


In [ ]:
USE DATABASE SKYPOINTS_INCU;

DESC TABLE OPS.REJECTED_REDEMPTION_RECORDS;

In [ ]:
-- CELL 5: OPERATIONAL AUDIT AND BATCH CONTROL

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

BEGIN TRANSACTION;

-- 1. INGESTION AUDIT
MERGE INTO OPS.INGESTION_AUDIT T
USING (
    SELECT
        '75e747f3112fe0f40d2adaaecf4cfa3f' AS BATCH_ID,
        'MEMBER' AS SOURCE_TYPE,
        'MEMBER_PROFILE_20261125_193100.csv' AS SOURCE_FILE_NAME,
        MAX(FILE_TIMESTAMP) AS FILE_TIMESTAMP,
        COUNT(*) AS RECORD_COUNT,
        COUNT(*) AS SUCCESS_COUNT,
        0 AS REJECTED_COUNT,
        'SUCCESS' AS STATUS
    FROM RAW.RAW_MEMBER_PROFILES
    WHERE BATCH_ID = '75e747f3112fe0f40d2adaaecf4cfa3f'

    UNION ALL

    SELECT
        '36dd50fa5adc4db81a373c1e7ec75cf1',
        'REDEMPTION',
        'REDEMPTION_20261125_202700.json',
        MAX(FILE_TIMESTAMP),
        COUNT(*),
        COUNT(*),
        0,
        'SUCCESS'
    FROM RAW.RAW_REDEMPTIONS
    WHERE BATCH_ID = '36dd50fa5adc4db81a373c1e7ec75cf1'
) S
ON T.BATCH_ID = S.BATCH_ID
AND T.SOURCE_TYPE = S.SOURCE_TYPE
AND T.SOURCE_FILE_NAME = S.SOURCE_FILE_NAME

WHEN MATCHED THEN UPDATE SET
    T.FILE_TIMESTAMP = S.FILE_TIMESTAMP,
    T.RECORD_COUNT = S.RECORD_COUNT,
    T.SUCCESS_COUNT = S.SUCCESS_COUNT,
    T.REJECTED_COUNT = S.REJECTED_COUNT,
    T.STATUS = S.STATUS,
    T.ERROR_MESSAGE = NULL

WHEN NOT MATCHED THEN INSERT (
    BATCH_ID, SOURCE_TYPE, SOURCE_FILE_NAME,
    FILE_TIMESTAMP, RECORD_COUNT, SUCCESS_COUNT,
    REJECTED_COUNT, STATUS
)
VALUES (
    S.BATCH_ID, S.SOURCE_TYPE, S.SOURCE_FILE_NAME,
    S.FILE_TIMESTAMP, S.RECORD_COUNT, S.SUCCESS_COUNT,
    S.REJECTED_COUNT, S.STATUS
);


-- 2. BATCH CONTROL
MERGE INTO OPS.BATCH_CONTROL T
USING (
    SELECT
        '75e747f3112fe0f40d2adaaecf4cfa3f' AS BATCH_ID,
        'MEMBER' AS SOURCE_TYPE,
        'MEMBER_PROFILE_20261125_193100.csv' AS SOURCE_FILE_NAME,
        MAX(FILE_TIMESTAMP) AS FILE_TIMESTAMP,
        COUNT(*) AS RECORD_COUNT,
        COUNT_IF(VALIDATION_STATUS = 'VALID') AS PROCESSED_COUNT,
        COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS REJECTED_COUNT
    FROM STG.STG_MEMBER_PROFILES
    WHERE BATCH_ID = '75e747f3112fe0f40d2adaaecf4cfa3f'

    UNION ALL

    SELECT
        '36dd50fa5adc4db81a373c1e7ec75cf1',
        'REDEMPTION',
        'REDEMPTION_20261125_202700.json',
        MAX(FILE_TIMESTAMP),
        COUNT(*),
        COUNT_IF(VALIDATION_STATUS = 'VALID'),
        COUNT_IF(VALIDATION_STATUS = 'REJECTED')
    FROM STG.STG_REDEMPTION_TRANSACTIONS
    WHERE BATCH_ID = '36dd50fa5adc4db81a373c1e7ec75cf1'
) S
ON T.BATCH_ID = S.BATCH_ID

WHEN MATCHED THEN UPDATE SET
    T.SOURCE_TYPE = S.SOURCE_TYPE,
    T.SOURCE_FILE_NAME = S.SOURCE_FILE_NAME,
    T.FILE_TIMESTAMP = S.FILE_TIMESTAMP,
    T.BATCH_END_TIMESTAMP = CURRENT_TIMESTAMP(),
    T.STATUS = 'SUCCESS',
    T.RECORD_COUNT = S.RECORD_COUNT,
    T.PROCESSED_COUNT = S.PROCESSED_COUNT,
    T.REJECTED_COUNT = S.REJECTED_COUNT

WHEN NOT MATCHED THEN INSERT (
    BATCH_ID, SOURCE_TYPE, SOURCE_FILE_NAME,
    FILE_TIMESTAMP, BATCH_START_TIMESTAMP,
    BATCH_END_TIMESTAMP, STATUS, RECORD_COUNT,
    PROCESSED_COUNT, REJECTED_COUNT
)
VALUES (
    S.BATCH_ID, S.SOURCE_TYPE, S.SOURCE_FILE_NAME,
    S.FILE_TIMESTAMP, CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(), 'SUCCESS', S.RECORD_COUNT,
    S.PROCESSED_COUNT, S.REJECTED_COUNT
);


-- 3. LOG REJECTED MEMBER RECORDS
INSERT INTO OPS.REJECTED_MEMBER_RECORDS (
    BATCH_ID, SOURCE_FILE_NAME, SOURCE_ROW_NUMBER,
    RAW_RECORD, REJECTION_REASON
)
SELECT
    S.BATCH_ID,
    S.SOURCE_FILE_NAME,
    S.SOURCE_ROW_NUMBER,
    R.RAW_RECORD,
    S.REJECTION_REASON
FROM STG.STG_MEMBER_PROFILES S
JOIN RAW.RAW_MEMBER_PROFILES R
    ON S.SOURCE_FILE_NAME = R.SOURCE_FILE_NAME
   AND S.SOURCE_ROW_NUMBER = R.SOURCE_ROW_NUMBER
WHERE S.VALIDATION_STATUS = 'REJECTED'
  AND NOT EXISTS (
      SELECT 1
      FROM OPS.REJECTED_MEMBER_RECORDS X
      WHERE X.BATCH_ID = S.BATCH_ID
        AND X.SOURCE_ROW_NUMBER = S.SOURCE_ROW_NUMBER
  );


-- 4. LOG REJECTED REDEMPTION RECORDS
INSERT INTO OPS.REJECTED_REDEMPTION_RECORDS (
    BATCH_ID, SOURCE_FILE_NAME, SOURCE_RECORD_NUMBER,
    RAW_RECORD, REJECTION_REASON
)
SELECT
    S.BATCH_ID,
    S.SOURCE_FILE_NAME,
    S.SOURCE_RECORD_NUMBER,
    R.RAW_RECORD,
    S.REJECTION_REASON
FROM STG.STG_REDEMPTION_TRANSACTIONS S
JOIN RAW.RAW_REDEMPTIONS R
    ON S.SOURCE_FILE_NAME = R.SOURCE_FILE_NAME
   AND S.SOURCE_RECORD_NUMBER = R.SOURCE_RECORD_NUMBER
WHERE S.VALIDATION_STATUS = 'REJECTED'
  AND NOT EXISTS (
      SELECT 1
      FROM OPS.REJECTED_REDEMPTION_RECORDS X
      WHERE X.BATCH_ID = S.BATCH_ID
        AND X.SOURCE_RECORD_NUMBER = S.SOURCE_RECORD_NUMBER
        AND X.REJECTION_REASON = S.REJECTION_REASON
  );

COMMIT;


-- 5. VERIFY INGESTION AUDIT
SELECT *
FROM OPS.INGESTION_AUDIT
ORDER BY INGESTION_TIMESTAMP DESC;


-- 6. VERIFY BATCH CONTROL
SELECT *
FROM OPS.BATCH_CONTROL
ORDER BY BATCH_START_TIMESTAMP DESC;


-- 7. VERIFY REJECTION COUNTS
SELECT
    'MEMBER' AS SOURCE_TYPE,
    COUNT(*) AS REJECTED_RECORDS
FROM OPS.REJECTED_MEMBER_RECORDS

UNION ALL

SELECT
    'REDEMPTION',
    COUNT(*)
FROM OPS.REJECTED_REDEMPTION_RECORDS;

In [ ]:
-- Verify ingestion audit
SELECT
    SOURCE_TYPE,
    SOURCE_FILE_NAME,
    RECORD_COUNT,
    SUCCESS_COUNT,
    REJECTED_COUNT,
    STATUS
FROM OPS.INGESTION_AUDIT
ORDER BY INGESTION_TIMESTAMP DESC;



In [ ]:
-- Verify batch processing control
SELECT
    SOURCE_TYPE,
    SOURCE_FILE_NAME,
    RECORD_COUNT,
    PROCESSED_COUNT,
    REJECTED_COUNT,
    STATUS
FROM OPS.BATCH_CONTROL
ORDER BY BATCH_START_TIMESTAMP DESC;

In [ ]:
%%sql -r dataframe_13
USE DATABASE SKYPOINTS_INCU;

SELECT
    'RAW_REDEMPTIONS' AS SOURCE_TABLE,
    COUNT(*) AS ACTUAL_RAW_COUNT
FROM RAW.RAW_REDEMPTIONS
WHERE BATCH_ID = '36dd50fa5adc4db81a373c1e7ec75cf1'

UNION ALL

SELECT
    'INGESTION_AUDIT' AS SOURCE_TABLE,
    RECORD_COUNT
FROM OPS.INGESTION_AUDIT
WHERE BATCH_ID = '36dd50fa5adc4db81a373c1e7ec75cf1'
  AND SOURCE_TYPE = 'REDEMPTION'

UNION ALL

SELECT
    'BATCH_CONTROL' AS SOURCE_TABLE,
    RECORD_COUNT
FROM OPS.BATCH_CONTROL
WHERE BATCH_ID = '36dd50fa5adc4db81a373c1e7ec75cf1';